In [1]:
import torch

data = torch.load("outputs/sae/analysis_z_con.pt", weights_only=False)

In [2]:
print(data.keys())
# dict_keys(['stats', 'correlations', 'exemplars', 'cooccurrence',
#  'top_cooccurring_pairs', 'decoder_similarity', 'decoder_clusters',
#  'probe_alignment', 'element_enrichment', 'dashboard', 'n_dead',
#  'n_rare', 'H_val', 'history'])

# Accessing each part

# 1. Activation stats — all ndarray (n_features,)
data["stats"]["fire_rate"]       # fraction of crystals activating each feature
data["stats"]["mean_act"]        # mean activation when active
data["stats"]["std_act"]         # std when active
data["stats"]["max_act"]         # max activation


dict_keys(['stats', 'correlations', 'exemplars', 'cooccurrence', 'top_cooccurring_pairs', 'decoder_similarity', 'decoder_clusters', 'probe_alignment', 'element_enrichment', 'dashboard', 'n_dead', 'n_rare', 'sim_threshold', 'variance_explained', 'H_val', 'history'])


array([3.9723477, 1.6096208, 1.2055223, ..., 1.4748013, 1.3878103,
       1.2401125], dtype=float32)

In [3]:
data["correlations"]["band_gap"]          # Pearson r for each feature vs band gap
data["correlations"]["formation_energy"]  # etc.

# 3. Top exemplars — dict[int, list[int]]
data["exemplars"][347]   # crystal indices that most activate feature 347

# 4. Co-occurrence — ndarray (n_features, n_features)
data["cooccurrence"][12, 347]  # fraction of crystals where both 12 and 347 fire

# Top pairs as a list of (feat_i, feat_j, rate)
data["top_cooccurring_pairs"][:5]

# 5. Decoder similarity — ndarray (n_features, n_features)
data["decoder_similarity"][12, 347]  # cosine sim between feature directions

# Clusters of similar features
data["decoder_clusters"]  # list of lists, e.g. [[12, 347, 901], [55, 2003]]

# 6. Probe alignment (if probe weights were provided)
data["probe_alignment"]["band_gap"]  # cosine sim between each SAE feature and probe direction

# 7. Element enrichment (if dataset was provided)
data["element_enrichment"][347]  # [("O", 3.2), ("Ti", 2.8), ...] — enrichment ratios

# 8. Dashboard — list of dicts, top 30 features by correlation
for feat in data["dashboard"][:5]:
    print(f"Feature {feat['feature_idx']}: "
        f"{feat['top_property']} r={feat['top_corr']:.3f}, "
        f"fires {feat['fire_rate']*100:.1f}%")

# 9. Sparse activations for all val crystals — Tensor (N_val, n_features)
data["H_val"].shape  # e.g. (9047, 4096)

# 10. Training history
data["history"]["train_loss"]     # list of floats, one per epoch
data["history"]["var_explained"]  # list of floats, one per epoch

Feature 3479: formation_energy r=-0.693, fires 20.2%
Feature 482: formation_energy r=-0.640, fires 15.6%
Feature 2871: num_atoms r=0.558, fires 23.9%
Feature 3430: formation_energy r=-0.527, fires 15.5%
Feature 684: formation_energy r=-0.525, fires 15.5%


[-1.6502899024499689,
 -1.43056160301066,
 -1.2398968439315237,
 -1.075645170302212,
 -0.9324994502847754,
 -0.8072085554230541,
 -0.6972886564287935,
 -0.6010016950019406,
 -0.514659010432954,
 -0.43787421545028926,
 -0.3680260986246977,
 -0.3057128446950894,
 -0.24851906656301614,
 -0.19624254866768576,
 -0.1483163407968089,
 -0.10402901749743254,
 -0.06305516726711202,
 -0.025042001727480834,
 0.01040899932665329,
 0.04384350680856686,
 0.07575480224624775,
 0.10571325534822829,
 0.13424369326997987,
 0.1616707186517038,
 0.1882461987518882,
 0.2137724600823372,
 0.2382867103062365,
 0.26196291702604,
 0.28498867406199047,
 0.30704752961649107,
 0.32832465092305496,
 0.34856107762352295,
 0.3682474157784633,
 0.387340546808298,
 0.40546770965927115,
 0.42269180892583214,
 0.43974884702909256,
 0.455893364081713,
 0.4713991288429201,
 0.48612605259421404,
 0.5006034423434904,
 0.5144186227963015,
 0.5275677051918628,
 0.5404248816478987,
 0.5528048916006376,
 0.5647302898626291,
 0.5

In [4]:
import numpy as np

corrs = data["correlations"]["band_gap"]
top_10 = np.argsort(np.abs(corrs))[::-1][:10]

for idx in top_10:
    print(f"  Feature {idx:4d}  r={corrs[idx]:+.4f}  "
        f"fire_rate={data['stats']['fire_rate'][idx]*100:.1f}%")

# %%
# ---- Step 1: Pick a feature to investigate ----
# Start with the dashboard (top 30 by correlation)
for feat in data["dashboard"]:
    elems = ""
    if "enriched_elements" in feat and feat["enriched_elements"]:
        elems = ", ".join(f"{s}({r:.1f}x)" for s, r in feat["enriched_elements"][:3])
    print(
        f"Feature {feat['feature_idx']:4d}  "
        f"fire={feat['fire_rate']*100:5.1f}%  "
        f"top={feat['top_property']:>20s} r={feat['top_corr']:+.3f}  "
        f"elems=[{elems}]"
    )

  Feature  482  r=+0.4606  fire_rate=15.6%
  Feature  684  r=+0.4488  fire_rate=15.5%
  Feature 1811  r=+0.3966  fire_rate=5.8%
  Feature 3479  r=+0.3932  fire_rate=20.2%
  Feature  575  r=+0.3661  fire_rate=18.9%
  Feature 2201  r=+0.3429  fire_rate=20.6%
  Feature 3440  r=+0.3187  fire_rate=22.1%
  Feature 2454  r=+0.3155  fire_rate=18.8%
  Feature 3500  r=+0.3075  fire_rate=17.3%
  Feature 3430  r=+0.3001  fire_rate=15.5%
Feature 3479  fire= 20.2%  top=    formation_energy r=-0.693  elems=[Ti(20.4x), Sr(4.0x), Sc(4.0x)]
Feature  482  fire= 15.6%  top=    formation_energy r=-0.640  elems=[Y(9.2x), Pa(8.6x), Lu(7.9x)]
Feature 2871  fire= 23.9%  top=           num_atoms r=+0.558  elems=[B(5.4x), Nd(5.4x), La(4.7x)]
Feature 3430  fire= 15.5%  top=    formation_energy r=-0.527  elems=[Pa(8.6x), Np(7.5x), U(5.9x)]
Feature  684  fire= 15.5%  top=    formation_energy r=-0.525  elems=[F(14.1x), Be(10.6x), V(8.2x)]
Feature 2201  fire= 20.6%  top=    formation_energy r=-0.511  elems=[Pu(8.2x),

In [ ]:
from pathlib import Path
import csv

def inspect_feature(feat_idx, data, csv_path="data/mp_20/val.csv", out_file="features/feature_report.txt"):
    stats = data["stats"]
    corrs = data["correlations"]

    def log(s=""):
        print(s)
        with open(out_file, "a") as f:
            f.write(s + "\n")

    log(f"\n{'='*60}")
    log(f"FEATURE {feat_idx}")
    log(f"{'='*60}")
    log(f"Fire rate:  {stats['fire_rate'][feat_idx]*100:.1f}%")
    log(f"Mean act:   {stats['mean_act'][feat_idx]:.4f}")
    log(f"Std act:    {stats['std_act'][feat_idx]:.4f}")
    log(f"Max act:    {stats['max_act'][feat_idx]:.4f}")
    log(f"\nProperty correlations:")
    for prop in sorted(corrs.keys()):
        r = corrs[prop][feat_idx]
        bar = "█" * int(abs(r) * 30)
        log(f"  {prop:>25s}  r={r:+.4f}  {bar}")
    if data["probe_alignment"] is not None:
        log(f"\nProbe direction alignment:")
        for prop, vals in data["probe_alignment"].items():
            log(f"  {prop:>25s}  cos={vals[feat_idx]:+.4f}")
    if data["element_enrichment"] is not None and feat_idx in data["element_enrichment"]:
        elems = data["element_enrichment"][feat_idx]
        if elems:
            log(f"\nEnriched elements (vs dataset baseline):")
            for sym, ratio in elems:
                log(f"  {sym:>3s}  {ratio:.2f}x")
    cooc = data["cooccurrence"]
    cooc_row = cooc[feat_idx].copy()
    cooc_row[feat_idx] = 0
    top_cooc = np.argsort(cooc_row)[::-1][:5]
    log(f"\nTop co-occurring features:")
    for j in top_cooc:
        log(f"  Feature {j:4d}  co-occur={cooc_row[j]*100:.1f}%")
    exemplars = data["exemplars"].get(feat_idx, [])
    if exemplars and Path(csv_path).exists():
        rows = list(csv.DictReader(open(csv_path)))
        log(f"\nTop activating crystals:")
        log(f"  {'Index':>6s}  {'Formula':>15s}  {'SG':>4s}  {'Ef':>8s}  {'Eg':>6s}  {'Ehull':>6s}")
        for ci in exemplars[:10]:
            if ci < len(rows):
                r = rows[ci]
                log(
                    f"  {ci:6d}  {r['pretty_formula']:>15s}  "
                    f"{r['spacegroup.number']:>4s}  "
                    f"{float(r['formation_energy_per_atom']):>8.3f}  "
                    f"{float(r['band_gap']):>6.3f}  "
                    f"{float(r['e_above_hull']):>6.3f}"
                )

In [ ]:
# Clear the report file before writing (prevents duplicates on re-run)
open("features/feature_report.txt", "w").close()

for i in range(len(data["dashboard"])):
    inspect_feature(data["dashboard"][i]["feature_idx"], data)

## Variance Explained by Feature Subsets

How much of the original activation variance do the dashboard's top features actually reconstruct?
This uses `SAEAnalyser.variance_explained_by_features` — pass any list of feature indices
to measure their joint (and optionally per-feature marginal) variance explained.

In [7]:
from mapping_exported import feature_labels, feature_to_cluster, label_to_cluster, join_table

In [8]:
# Load the SAE model and original activations needed for variance computation
from src.interpretability.sae_model import TopKSAE, SAEConfig
from src.interpretability.analyse_sae import SAEAnalyser

sae_ckpt = torch.load("outputs/sae/sae_z_con.pt", weights_only=False)
sae = TopKSAE(sae_ckpt["config"])
sae.load_state_dict(sae_ckpt["state_dict"])
sae.eval()

analyser = SAEAnalyser(sae, device="cpu")

# Original activations (what the SAE reconstructs)
acts = torch.load("outputs/activations_val.pt", weights_only=False)
X_val = acts["z_con"].float()

# Sparse SAE activations (already saved in analysis)
H = data["H_val"]

print(f"X_val: {X_val.shape}  H: {H.shape}")
print(f"Input norm — μ range: [{sae.input_mean.min():.3f}, {sae.input_mean.max():.3f}]")
print(f"Input norm — σ range: [{sae.input_std.min():.3f}, {sae.input_std.max():.3f}]")
print(f"Overall SAE var explained (normalized): {data['history']['var_explained'][-1]:.4f}")
if 'var_explained_raw' in data['history']:
    print(f"Overall SAE var explained (raw):        {data['history']['var_explained_raw'][-1]:.4f}")

X_val: torch.Size([9047, 256])  H: torch.Size([9047, 4096])
Input norm — μ range: [-0.781, 0.366]
Input norm — σ range: [0.342, 1.430]
Overall SAE var explained (normalized): 0.8455
Overall SAE var explained (raw):        0.8636


In [9]:
# --- Dashboard features: joint + per-feature marginal VE ---

dashboard_indices = [e["feature_idx"] for e in data["dashboard"]]
n_dash = len(dashboard_indices)

ve = analyser.variance_explained_by_features(
    X_val, H, dashboard_indices, per_feature=True,
)

print(f"Dashboard features ({n_dash} / {H.shape[1]} total)")
print(f"  Joint variance explained: {ve['joint']:.4f}")
print(f"  Overall SAE VE:           {data['history']['var_explained'][-1]:.4f}")
print(f"  Ratio (dashboard / all):  {ve['joint'] / data['history']['var_explained'][-1]:.2%}")

print(f"\nPer-feature marginal VE (top 15 contributors):")
print(f"  {'Feat':>5s}  {'Marginal VE':>11s}  {'Fire%':>6s}  {'TopCorr':>8s}  Label")
print(f"  {'-'*65}")

# Sort marginals descending
sorted_marginals = sorted(ve["marginals"].items(), key=lambda kv: kv[1], reverse=True)

# Build a quick lookup from dashboard entries
dash_lookup = {e["feature_idx"]: e for e in data["dashboard"]}

for feat_idx, mve in sorted_marginals[:15]:
    entry = dash_lookup[feat_idx]
    print(
        f"  {feat_idx:5d}  {mve:11.4f}  {entry['fire_rate']*100:5.1f}%  "
        f"{entry['top_corr']:+7.4f}  {entry['top_property']}  | {feature_labels.get(feat_idx, 'unlabelled')}"
    )

Dashboard features (100 / 4096 total)
  Joint variance explained: 0.6510
  Overall SAE VE:           0.8455
  Ratio (dashboard / all):  77.00%

Per-feature marginal VE (top 15 contributors):
   Feat  Marginal VE   Fire%   TopCorr  Label
  -----------------------------------------------------------------
   2262       0.0497   27.2%  +0.2612  formation_energy  | Ca/Si/Mg silicate compound
   3500       0.0495   17.3%  +0.3075  band_gap  | Rb/Cl/K alkali halide
    135       0.0489   24.3%  -0.3001  spacegroup  | I/B/Se chalcogenide halide
    418       0.0482   23.9%  +0.3486  formation_energy  | Np/B/Al metallic intermetallic
   2174       0.0481   18.0%  +0.3658  crystal_system  | Er/Hf/Tm metallic compound
   2105       0.0476   23.2%  +0.2869  is_metal  | C/Re/Si carbide/silicide
   3374       0.0475   21.5%  +0.2683  is_metal  | Ni/Al intermetallic compound
   1449       0.0474   23.2%  +0.3292  formation_energy  | Actinide/Au low-symmetry metal
   2800       0.0465   24.5%  -0.233

In [10]:
# --- Incremental VE: how does VE grow as we add dashboard features one by one? ---
# Features are added in dashboard order (descending |correlation|)

incremental_ve = []
for k in range(1, n_dash + 1):
    subset = dashboard_indices[:k]
    result = analyser.variance_explained_by_features(X_val, H, subset)
    incremental_ve.append(result["joint"])

print(f"Incremental VE as dashboard features are added (by |correlation| rank):")
print(f"  {'N':>4s}  {'Joint VE':>9s}  {'Δ VE':>8s}  Feature added")
print(f"  {'-'*50}")
for k in range(min(n_dash, 20)):
    delta = incremental_ve[k] - (incremental_ve[k-1] if k > 0 else 0)
    feat = data["dashboard"][k]
    print(
        f"  {k+1:4d}  {incremental_ve[k]:9.4f}  {delta:+8.4f}  "
        f"#{feat['feature_idx']} ({feat['top_property']} r={feat['top_corr']:+.3f})"
    )

if n_dash > 20:
    print(f"  ...")
    for k in [n_dash-1]:
        delta = incremental_ve[k] - incremental_ve[k-1]
        feat = data["dashboard"][k]
        print(
            f"  {k+1:4d}  {incremental_ve[k]:9.4f}  {delta:+8.4f}  "
            f"#{feat['feature_idx']} ({feat['top_property']} r={feat['top_corr']:+.3f})"
        )

Incremental VE as dashboard features are added (by |correlation| rank):
     N   Joint VE      Δ VE  Feature added
  --------------------------------------------------
     1     0.0404   +0.0404  #3479 (formation_energy r=-0.693)
     2     0.0476   +0.0072  #482 (formation_energy r=-0.640)
     3     0.0602   +0.0126  #2871 (num_atoms r=+0.558)
     4     0.0661   +0.0059  #3430 (formation_energy r=-0.527)
     5     0.0769   +0.0108  #684 (formation_energy r=-0.525)
     6     0.0890   +0.0121  #2201 (formation_energy r=-0.511)
     7     0.1012   +0.0123  #2708 (formation_energy r=-0.508)
     8     0.1069   +0.0056  #3429 (formation_energy r=-0.481)
     9     0.1100   +0.0031  #2421 (formation_energy r=-0.478)
    10     0.1180   +0.0080  #2454 (num_atoms r=+0.476)
    11     0.1277   +0.0097  #3850 (formation_energy r=-0.440)
    12     0.1369   +0.0092  #1862 (formation_energy r=-0.434)
    13     0.1395   +0.0025  #3519 (formation_energy r=-0.432)
    14     0.1417   +0.0022  

In [11]:
from collections import defaultdict

# rebuild labelled_clusters from feature_to_cluster
labelled_clusters = defaultdict(list)
for fid, cluster in feature_to_cluster.items():
    labelled_clusters[cluster].append(fid)
labelled_clusters = dict(labelled_clusters)

print(f"Variance explained by labelled cluster:")
print(f"  {'Cluster':<30s}  {'N':>3s}  {'Joint VE':>9s}")
print(f"  {'-'*50}")
all_labelled = []
cluster_ves = {}
for name, indices in labelled_clusters.items():
    result = analyser.variance_explained_by_features(X_val, H, indices)
    cluster_ves[name] = result["joint"]
    all_labelled.extend(indices)
    print(f"  {name:<30s}  {len(indices):3d}  {result['joint']:9.4f}")

unique_labelled = list(set(all_labelled))
all_ve = analyser.variance_explained_by_features(X_val, H, unique_labelled)
print(f"  {'-'*50}")
print(f"  {'ALL labelled':<30s}  {len(unique_labelled):3d}  {all_ve['joint']:9.4f}")
print(f"  {'Overall SAE':<30s}  {H.shape[1]:3d}  {data['history']['var_explained'][-1]:9.4f}")

Variance explained by labelled cluster:
  Cluster                           N   Joint VE
  --------------------------------------------------
  Early transition-metal oxides/fluorides   18     0.1532
  Structurally complex/low-symmetry    6     0.1065
  Actinide compounds                7     0.0743
  Heavy complex oxides              6     0.0605
  Carbides/silicides/borides        4     0.0722
  Chalcogenides                     5     0.0539
  Alkali/alkaline-earth halides    12     0.1063
  Rare-earth/lanthanide compounds   10     0.0919
  Noble/platinum-group metals       9     0.1129
  Refractory transition metals      8     0.1028
  Intermetallics and post-transition metals    6     0.0891
  Polyanionic networks              7     0.0773
  --------------------------------------------------
  ALL labelled                     98     0.6464
  Overall SAE                     4096     0.8455


In [12]:
# --- Rare vs common features VE ---
# Do rare features (< 1% fire rate) carry meaningful reconstruction signal?

fire_rates = data["stats"]["fire_rate"]
rare_idx  = [j for j in range(H.shape[1]) if 0 < fire_rates[j] < 0.01]
common_idx = [j for j in range(H.shape[1]) if fire_rates[j] >= 0.10]
dead_idx  = [j for j in range(H.shape[1]) if fire_rates[j] == 0]

rare_ve   = analyser.variance_explained_by_features(X_val, H, rare_idx)   if rare_idx   else {"joint": 0.0}
common_ve = analyser.variance_explained_by_features(X_val, H, common_idx) if common_idx else {"joint": 0.0}
dead_ve   = analyser.variance_explained_by_features(X_val, H, dead_idx)   if dead_idx   else {"joint": 0.0}

print(f"Variance explained by feature activity level:")
print(f"  {'Group':<25s}  {'Count':>5s}  {'Joint VE':>9s}  {'VE/feat':>9s}")
print(f"  {'-'*55}")
for name, idx, result in [
    ("Dead (0% fire rate)",    dead_idx,   dead_ve),
    ("Rare (0-1% fire rate)",  rare_idx,   rare_ve),
    ("Common (≥10% fire rate)", common_idx, common_ve),
]:
    n = len(idx)
    per_feat = result["joint"] / n if n > 0 else 0
    print(f"  {name:<25s}  {n:5d}  {result['joint']:9.4f}  {per_feat:9.6f}")

print(f"\n  Dead features VE should be ~0 (sanity check): {dead_ve['joint']:.6f}")

Variance explained by feature activity level:
  Group                      Count   Joint VE    VE/feat
  -------------------------------------------------------
  Dead (0% fire rate)            0     0.0000   0.000000
  Rare (0-1% fire rate)       3823     0.1075   0.000028
  Common (≥10% fire rate)      100     0.7642   0.007642

  Dead features VE should be ~0 (sanity check): 0.000000


In [13]:
# --- Decoder similarity clusters: redundancy check ---
# If features in a cluster all point in similar directions, do they explain
# much more variance together than a single member alone?

dec_clusters = data["decoder_clusters"]
if dec_clusters:
    print(f"Decoder similarity clusters — redundancy check:")
    print(f"  {'Cluster':>7s}  {'Size':>4s}  {'Joint VE':>9s}  {'Best single':>11s}  {'Ratio':>6s}")
    print(f"  {'-'*50}")

    for i, members in enumerate(dec_clusters[:10]):
        joint = analyser.variance_explained_by_features(X_val, H, members)
        single = analyser.variance_explained_by_features(X_val, H, members, per_feature=True)
        best_single_ve = max(single["marginals"].values())
        ratio = joint["joint"] / best_single_ve if best_single_ve > 0 else float("inf")
        print(
            f"  {i:7d}  {len(members):4d}  {joint['joint']:9.4f}  "
            f"{best_single_ve:11.4f}  {ratio:6.2f}x"
        )
    print(f"\n  Ratio ≈ 1 means the cluster is redundant (single feature captures it all)")
    print(f"  Ratio >> 1 means the features contribute complementary variance")
else:
    print("No decoder similarity clusters found.")

No decoder similarity clusters found.
